In [2]:
import pandas as pd

articles = pd.read_csv('./articles.csv')
customers = pd.read_csv('./customers.csv')
transactions = pd.read_csv('./transactions_train.csv')

In [3]:
articles['text'] = (
    articles['prod_name'] + ' ' +
    articles['product_type_name'] + ' ' +
    articles['colour_group_name'] + ' ' +
    articles['garment_group_name'] + ' ' +
    articles['detail_desc'].fillna('')
)

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('all-MiniLM-L6-v2',device='cuda')
all_texts = articles['text'].tolist()
#all_embeddings = st_model.encode(all_texts, show_progress_bar=True, batch_size=256)
#np.save('article_embeddings.npy', all_embeddings)
all_embeddings = np.load('article_embeddings.npy')

c:\Users\swson23\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [5]:
import faiss
dimension = all_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(all_embeddings.astype('float32'))


######여기까지 임베딩 벡터########

In [6]:
query = "casual summer white t-shirt"
query_embedding = st_model.encode([query])

D, I = index.search(query_embedding.astype('float32'), k=10)

# 중복 제거
seen = set()
results = []
for idx in I[0]:
    prod_name = articles.iloc[idx]['prod_name']
    if prod_name not in seen:
        seen.add(prod_name)
        results.append(idx)
    if len(results) == 5:
        break

print("검색 결과 (중복 제거):")
for i, idx in enumerate(results):
    print(f"{i+1}. {articles.iloc[idx]['prod_name']} - {articles.iloc[idx]['detail_desc']}")

검색 결과 (중복 제거):
1. Summer price tee - T-shirt in soft cotton jersey with a print motif.
2. SG Summer Top - Wide-fitting sports top in fast-drying functional fabric with a print motif on the front, cap sleeves and a longer, sewn-in vest top with a racer back.
3. SUMMER Top - Short, wide, sports top in printed, fast-drying mesh layered over a vest top with a racer back.
4. Summer top - Short, wide, sports top in printed, fast-drying mesh layered over a vest top with a racer back.
5. Summer graphic tee - T-shirt in soft jersey with a motif on the front.


In [7]:
# 구매 많은 유저 한 명 뽑기
top_user = transactions['customer_id'].value_counts().index[0]
user_history = transactions[transactions['customer_id'] == top_user]['article_id'].tolist()

In [8]:
# 유저 히스토리 임베딩 평균
article_id_to_idx = {aid: idx for idx, aid in enumerate(articles['article_id'])}

user_embeddings = []
for aid in user_history:
    if aid in article_id_to_idx:
        idx = article_id_to_idx[aid]
        user_embeddings.append(all_embeddings[idx])

user_vector = np.mean(user_embeddings, axis=0, keepdims=True)

# 추천
D, I = index.search(user_vector.astype('float32'), k=10)

seen = set(user_history)
results = []
for idx in I[0]:
    aid = articles.iloc[idx]['article_id']
    if aid not in seen:
        results.append(idx)
    if len(results) == 5:
        break

print("유저 맞춤 추천:")
for i, idx in enumerate(results):
    print(f"{i+1}. {articles.iloc[idx]['prod_name']} - {articles.iloc[idx]['detail_desc']}")
    
    ##############################

유저 맞춤 추천:
1. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.
2. Twiggy - Calf-length dress in mesh with a low-cut back and long sleeves. Seam at the waist, a gently flared skirt and raw edges at the cuffs and hem. Jersey lining.
3. LASSIE LINEN L/S - Long-sleeved top in a linen weave with a V-neck. Slightly longer at the back.
4. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.
5. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.


In [9]:
seen_names = set()
seen_ids = set(user_history)
results = []
for idx in I[0]:
    aid = articles.iloc[idx]['article_id']
    name = articles.iloc[idx]['prod_name']
    if aid not in seen_ids and name not in seen_names:
        seen_names.add(name)
        results.append(idx)
    if len(results) == 5:
        break

print("유저 맞춤 추천 (중복 제거):")
for i, idx in enumerate(results):
    print(f"{i+1}. {articles.iloc[idx]['prod_name']} - {articles.iloc[idx]['detail_desc']}")

유저 맞춤 추천 (중복 제거):
1. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.
2. Twiggy - Calf-length dress in mesh with a low-cut back and long sleeves. Seam at the waist, a gently flared skirt and raw edges at the cuffs and hem. Jersey lining.
3. RAVEN halfzip ls - Fitted running top in fast-drying functional fabric with a stand-up collar and zip at the top. Long sleeves with thumbholes at the cuffs, a small zipped pocket at the back, reflective details and a gently rounded hem. Slightly longer at the back. Soft brushed inside.
4. HEATHER halfzip ls - Fitted running top in fast-drying functional fabric with an elasticated hood and a zip at the top. Long raglan sleeves, a small zipped pocket at the back, reflective details and a gently rounded hem. Slightly longer at the back. Soft brushed inside.
5. ELENA baselayer - Base layer tights in a soft wool blend with an elasticated waist.


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "./qwen"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
def rerank_with_llm(query, candidates, top_k=3):
    candidate_text = "\n".join([
        f"{i+1}. {articles.iloc[idx]['prod_name']}: {articles.iloc[idx]['detail_desc']}"
        for i, idx in enumerate(candidates)
    ])
    
    prompt = f"""당신은 패션 추천 전문가입니다. 반드시 한국어로만 답변하세요. 한자나 중국어는 절대 사용하지 마세요.

사용자 검색어: "{query}"

후보 상품:
{candidate_text}

위 후보 중 가장 적합한 {top_k}개를 선택하고 이유를 설명하세요.
형식:
1. [상품 번호] - [이유]
2. [상품 번호] - [이유]
3. [상품 번호] - [이유]"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(qwen_model.device)
    outputs = qwen_model.generate(**inputs, max_new_tokens=200)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

In [12]:
query = "나한테 잘 맞는 옷"
query_embedding = st_model.encode([query])
D, I = index.search(query_embedding.astype('float32'), k=10)
result = rerank_with_llm(query, I[0][:5])
print(result)

1. 2 - [Mia l/s top] - 이 옷은 긴 가슴을 자랑하며, 여성스럽고 쾌적한 소재인 토털 섹시 라운드네일과 날카로운 핑크 컬러가 여성미를 더해줍니다. 긴袖는 겨울철에도 활용 가능하며, 기본적인 스타일에 쉽게 매치할 수 있습니다.
2. 3 - [W GARDA SKIRT EQ] - 이 옷은 높은 웨이스트와detachable tie belt가 있어 몸매를 살리는데 효과적이며, 편안하면서도 세련된 디자인이 특징입니다. 티셔츠나 티셔츠 위에 입을 수 있어 다양한 스타일링이 가능합니다.
3. 5 - [Kardashian skirt (


In [13]:
user_vector = np.mean(user_embeddings, axis=0, keepdims=True)

D, I = index.search(user_vector.astype('float32'), k=20)

seen_names = set()
seen_ids = set(user_history)
candidates = []
for idx in I[0]:
    aid = articles.iloc[idx]['article_id']
    name = articles.iloc[idx]['prod_name']
    if aid not in seen_ids and name not in seen_names:
        seen_names.add(name)
        candidates.append(idx)
    if len(candidates) == 5:
        break

result = rerank_with_llm("이 유저의 구매 히스토리 기반 추천", candidates)
print(result)

1. 3 - [Raven halfzip LS] - 이 상품은 사용자의 구매 히스토리에 따라 운동복 히트웨어를 추천하는 것이 적합합니다. 그 이유는 사용자가 이전에 같은 타입의 운동 상품을 구매한 경험이 있기 때문입니다.
2. 4 - [HEATHER halfzip LS] - 이 상품 역시 운동복과 관련된 제품으로, 사용자의 구매 패턴을 고려할 때 적합합니다. 특히 이 상품은 에어링 디테일과 립업 캡슐이 있어 더욱 세부적인 추천 요소가 될 수 있습니다.
3. 1 - [LASSIE LINEN L/S] - 마지막으로, 이 상품은 셔츠와 같은 페미닌한 스타일의 제품으로, 여성 고객의 구


In [14]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch
from datasets import load_dataset
dataset = load_dataset("beomi/KoAlpaca-v1.1a")
def format_data(item):
    return {"text": f"### 지시사항:\n{item['instruction']}\n\n### 답변:\n{item['output']}"}

train_dataset = dataset['train'].map(format_data, remove_columns=['instruction', 'output', 'url'])

tokenizer = AutoTokenizer.from_pretrained("./qwen")
tokenizer.pad_token = tokenizer.eos_token

# 토크나이징
def tokenize(item):
    result = tokenizer(
        item["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = train_dataset.map(tokenize, remove_columns=["text"])
print(tokenized_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 21155
})


In [15]:
import torch
from peft import LoraConfig, get_peft_model

qwen_model = AutoModelForCausalLM.from_pretrained(
    "./qwen",
    torch_dtype=torch.float16,
    device_map="cuda"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

#qwen_model = get_peft_model(qwen_model, lora_config)
#qwen_model.config.use_cache = False
#qwen_model.print_trainable_parameters()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [16]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./qwen_korean",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    save_steps=500,
    warmup_steps=100,
    report_to="none",
    save_safetensors=False
)

trainer = Trainer(
    model=qwen_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

#trainer.train()

In [17]:
#qwen_model.save_pretrained("./qwen_korean")
#tokenizer.save_pretrained("./qwen_korean")

In [18]:
# 1. 모델 로드
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("./qwen")
base_model = AutoModelForCausalLM.from_pretrained(
    "./qwen",
    torch_dtype=torch.float16,
    device_map="cuda"
)
korean_model = PeftModel.from_pretrained(base_model, "./qwen_korean")
#korean_model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [19]:
del qwen_model
del base_model
torch.cuda.empty_cache()

In [20]:
def rerank_with_llm(query, candidates, top_k=3):
    candidate_text = "\n".join([
        f"{i+1}. 상품명: {articles.iloc[idx]['prod_name']}, 카테고리: {articles.iloc[idx]['product_type_name']}, 색상: {articles.iloc[idx]['colour_group_name']}, 설명: {articles.iloc[idx]['detail_desc'][:80]}"
        for i, idx in enumerate(candidates)
    ])
    
    prompt = f"""당신은 패션 추천 전문가입니다. 반드시 한국어로만 답변하세요. 한자나 중국어는 절대 사용하지 마세요.

사용자 검색어: "{query}"

후보 상품:
{candidate_text}

위 후보 중 가장 적합한 {top_k}개를 선택하고 이유를 설명하세요.
형식:
[상품 번호] - [이유]
"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(korean_model.device)
    outputs = korean_model.generate(**inputs, max_new_tokens=500)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response


In [21]:
query = "나한테 잘 맞는 옷"
query_embedding = st_model.encode([query])
D, I = index.search(query_embedding.astype('float32'), k=10)
result = rerank_with_llm(query, I[0][:5])
print(result)

1. [상품명: W GARDA SKIRT EQ, 색상: Beige] - 높은 밸런스와 시원함을 느낄 수 있는 beige 색상의 스커트입니다. 허리가 높아지고, 부드러운 니트와 함께 착용하기 좋습니다.
2. [상품명: W GARDA SKIRT EQ, 색상: Pink] - 이전과 동일한 이유로 선택합니다. 다양한 스타일에 어울리는 색상으로, 여성스럽고 매력적인 이미지를 줍니다.
3. [상품명: Singapore, 색상: Dark Grey] - 디테일이 과한 것이 아닌 단순하면서도 우아한 디자인으로, 다양한 스타일에 어울릴 것입니다. 흰색이나 다른 색상과 함께 착용하기 좋은 편안한 느낌을 줍니다.

이 외에도, 사용자의 키, 몸매, 스타일에 따라 다른 스커트들이 더 적합할 수 있습니다. 하지만 이 세 가지는 대부분의 경우에 적합한 스커트입니다.
